# Text processing: a companion notebook

**Human Language Technologies 2026-27, Lecture 03.** Claudio Gallicchio, University of Pisa.

This notebook lets you run the examples of the lecture and check your answers to the exercise sheet. It follows Chapter 2 of Jurafsky and Martin, _Speech and Language Processing_ (3rd edition draft, August 2026), sections 2.6 to 2.9.

1. A small regular expression tester
2. Regular expressions, step by step (exercises 1, 2, 3, 5, 6 of the sheet)
3. Substitutions, capture groups and lookahead (exercise 4)
4. ELIZA, a cascade of substitutions (textbook exercise 2.6)
5. The GPT-2 pre-tokenizer (exercise 7)
6. Word counts, the Unix way
7. Rule-based tokenization and sentence segmentation (exercises 8 and 9)
8. Minimum edit distance and the backtrace (exercises 10 to 12, textbook exercises 2.9 and 2.10)

How to use it: do the exercises on paper first, then run the cells from top to bottom (Jupyter, VS Code or Google Colab) and compare. Only the Python standard library is needed. The cells marked **Optional** need something more: the `regex` module (`pip install regex`), an internet connection, or a Unix shell (macOS, Linux, Colab). Change the patterns and the strings, and look at what comes out.


## 1. A small regular expression tester

Two helpers used in the whole notebook.

- `show(pattern, text, ...)` highlights every match of the pattern in each text, as `re.finditer` finds them, from left to right. Consecutive matches alternate in color; a red bar marks a match of length zero (anchors, boundaries, lookahead).
- `check(pattern, accept, reject)` tests a pattern on strings that should match and strings that should not, and reports which ones it gets wrong. By default the whole string must match (`re.fullmatch`); with `how="search"` a match anywhere is enough (`re.search`).

Patterns are written as Python raw strings, `r"..."`, as in the book: backslashes reach the regex engine unchanged.


In [1]:
import re, html
from IPython.display import display

COLORS = [
    ("#D3E6F5", "#1D73A6"),
    ("#D5F0EC", "#139C8A"),
]  # (background, underline), alternating


class Matches:
    """All the matches of a pattern in a text, displayed with highlighting."""

    def __init__(self, pattern, text, flags=0):
        self.pattern, self.text = pattern, text
        self.spans = [m.span() for m in re.finditer(pattern, text, flags)]

    def _pieces(self):
        pos = 0
        for k, (a, b) in enumerate(self.spans):
            yield "plain", self.text[pos:a], k
            yield "match", self.text[a:b], k
            pos = b
        yield "plain", self.text[pos:], None

    def __repr__(self):  # plain text: matches in [brackets]
        body = "".join(
            f"[{s}]" if kind == "match" else s for kind, s, _ in self._pieces()
        )
        return f'r"{self.pattern}"   {body}   ({len(self.spans)} matches)'

    def _repr_html_(self):
        parts = []
        for kind, s, k in self._pieces():
            if kind == "plain":
                parts.append(html.escape(s))
            elif s == "":
                parts.append(
                    '<span style="border-left:3px solid #F06C5F;margin:0 1px"></span>'
                )
            else:
                bg, line = COLORS[k % 2]
                parts.append(
                    f'<span style="background:{bg};border-bottom:2px solid {line}">{html.escape(s)}</span>'
                )
        n = len(self.spans)
        return (
            '<div style="font-family:Courier New,monospace;font-size:15px;line-height:1.9;white-space:pre-wrap">'
            f'<span style="color:#173665">r"{html.escape(self.pattern)}"</span>   {"".join(parts)}'
            f'<span style="color:#596675;font-family:Arial">   {n} match{"" if n == 1 else "es"}</span></div>'
        )


def show(pattern, *texts, flags=0):
    for t in texts:
        display(Matches(pattern, t, flags))


def check(pattern, accept=(), reject=(), how="fullmatch", flags=0):
    find = {"fullmatch": re.fullmatch, "search": re.search}[how]
    rows = [(s, True) for s in accept] + [(s, False) for s in reject]
    wrong = 0
    print(f'r"{pattern}"   ({how})')
    for s, want in rows:
        got = find(pattern, s, flags) is not None
        wrong += got != want
        verdict = "ok   " if got == want else "WRONG"
        print(
            f"  {verdict}  {'should match    ' if want else 'should not match'}  {s!r}"
        )
    print(
        f"  {len(rows) - wrong} of {len(rows)} correct"
        + ("" if wrong else ": all good")
    )


show(r"Buttercup", "I'm called little Buttercup")
check(r"colou?r", accept=["color", "colour"], reject=["colouur", "colr"])

r"Buttercup"   I'm called little [Buttercup]   (1 matches)

r"colou?r"   (fullmatch)
  ok     should match      'color'
  ok     should match      'colour'
  ok     should not match  'colouur'
  ok     should not match  'colr'
  4 of 4 correct: all good


## 2. Regular expressions, step by step

### Character disjunction: square brackets

The examples of Figures 2.8 to 2.10 of the book. A bracket matches **one** character among those listed; a dash inside brackets gives a range; a caret right after the open bracket negates the set; elsewhere inside brackets it is just a caret. Figure 2.10 says the same of a caret in the middle of a pattern, but in Python a caret outside brackets is always an anchor: to match it, escape it.


In [2]:
show(r"[wW]oodchuck", "Woodchuck or woodchuck?")
show(r"[0-9]", "plenty of 7 to 5")
show(r"[A-Z]", "we should call it 'Drenched Blossoms'")
show(r"[^A-Z]", "Oyfn pripetchik")
show(r"[^Ss]", "I have no exquisite reason for't")
show(r"[e^]", "look up ^ now")
show(
    r"a^b", "look up a^b now"
)  # in Python a caret outside brackets is always the start-of-line anchor
show(r"a\^b", "look up a^b now")  # so a literal caret must be escaped

r"[wW]oodchuck"   [Woodchuck] or [woodchuck]?   (2 matches)

r"[0-9]"   plenty of [7] to [5]   (2 matches)

r"[A-Z]"   we should call it '[D]renched [B]lossoms'   (2 matches)

r"[^A-Z]"   O[y][f][n][ ][p][r][i][p][e][t][c][h][i][k]   (14 matches)

r"[^Ss]"   [I][ ][h][a][v][e][ ][n][o][ ][e][x][q][u][i]s[i][t][e][ ][r][e][a]s[o][n][ ][f][o][r]['][t]   (30 matches)

r"[e^]"   look up [^] now   (1 matches)

r"a^b"   look up a^b now   (0 matches)

r"a\^b"   look up [a^b] now   (1 matches)

### Repetition, counting and wildcards

Counters apply to the character (or parenthesized expression) immediately before them: `*` zero or more, `+` one or more, `?` zero or one, `{n}` exactly n, `{n,m}` from n to m. The period is a wildcard for any single character except a newline. The sheep language _baa!, baaa!, baaaa!, ..._ in three versions:


In [3]:
SHEEP = ["baa!", "baaa!", "baaaaaa!"]
NOT_SHEEP = ["ba!", "b!", "baa", "baab!"]
check(r"ba*!", SHEEP, NOT_SHEEP)  # wrong: the star allows zero a
check(r"baa*!", SHEEP, NOT_SHEEP)
check(r"baa+!", SHEEP, NOT_SHEEP)

r"ba*!"   (fullmatch)
  ok     should match      'baa!'
  ok     should match      'baaa!'
  ok     should match      'baaaaaa!'
  WRONG  should not match  'ba!'
  WRONG  should not match  'b!'
  ok     should not match  'baa'
  ok     should not match  'baab!'
  5 of 7 correct
r"baa*!"   (fullmatch)
  ok     should match      'baa!'
  ok     should match      'baaa!'
  ok     should match      'baaaaaa!'
  WRONG  should not match  'ba!'
  ok     should not match  'b!'
  ok     should not match  'baa'
  ok     should not match  'baab!'
  6 of 7 correct
r"baa+!"   (fullmatch)
  ok     should match      'baa!'
  ok     should match      'baaa!'
  ok     should match      'baaaaaa!'
  ok     should not match  'ba!'
  ok     should not match  'b!'
  ok     should not match  'baa'
  ok     should not match  'baab!'
  7 of 7 correct: all good


In [4]:
show(r"beg.n", "begin, beg'n, begun, began, beg\nn")
show(r"[0-9]+", "Room 101 costs $1250 a night")
show(r"\d{3}", "Room 101 costs $1250 a night")
show(r"[ab]*", "aaaa ababab bbbb")

r"beg.n"   [begin], [beg'n], [begun], [began], beg
n   (4 matches)

r"[0-9]+"   Room [101] costs $[1250] a night   (2 matches)

r"\d{3}"   Room [101] costs $[125]0 a night   (2 matches)

r"[ab]*"   [aaaa][] [ababab][] [bbbb][]   (6 matches)

In the last line, `[ab]*` also matches the empty string: at every position where no _a_ or _b_ follows, it succeeds with a match of length zero (the red bars).

### Exercise 1 of the sheet: reading patterns

Answer on paper first, then run the cell.


In [5]:
for s in ["color", "colour", "colouur"]:
    print(f"colou?r  {s!r:10}", "match" if re.fullmatch(r"colou?r", s) else "no match")
for s in ["ba!", "baa!", "baaaa!", "baaa"]:
    print(f"baa+!    {s!r:10}", "match" if re.fullmatch(r"baa+!", s) else "no match")
show(r"\b[tT]he\b", "The other cat sat there, then the dog left.")

colou?r  'color'    match
colou?r  'colour'   match
colou?r  'colouur'  no match
baa+!    'ba!'      no match
baa+!    'baa!'     match
baa+!    'baaaa!'   match
baa+!    'baaa'     no match


r"\b[tT]he\b"   [The] other cat sat there, then [the] dog left.   (2 matches)

### Anchors and boundaries

`^` and `$` anchor a pattern to the start and the end of the line; `\b` matches at a word boundary, `\B` anywhere else. A _word_, for `\b`, is a sequence of letters, digits and underscores. Anchors and boundaries match the empty string: they do not consume characters.


In [6]:
show(r"^The", "The dog saw The Cat.")
check(
    r"^The dog\.$",
    accept=["The dog."],
    reject=["The dog!", "The dogo", "The dog. Yes."],
)
check(
    r"^The dog.$", accept=["The dog."], reject=["The dog!", "The dogo", "The dog. Yes."]
)  # the period is a wildcard
show(r"\b99\b", "There are 99 bottles", "It costs $99", "There are 299 bottles")
show(r"\b", "the cat")

r"^The"   [The] dog saw The Cat.   (1 matches)

r"^The dog\.$"   (fullmatch)
  ok     should match      'The dog.'
  ok     should not match  'The dog!'
  ok     should not match  'The dogo'
  ok     should not match  'The dog. Yes.'
  4 of 4 correct: all good
r"^The dog.$"   (fullmatch)
  ok     should match      'The dog.'
  WRONG  should not match  'The dog!'
  WRONG  should not match  'The dogo'
  ok     should not match  'The dog. Yes.'
  2 of 4 correct


r"\b99\b"   There are [99] bottles   (1 matches)

r"\b99\b"   It costs $[99]   (1 matches)

r"\b99\b"   There are 299 bottles   (0 matches)

r"\b"   []the[] []cat[]   (4 matches)

### Disjunction, grouping and precedence

The pipe `|` is disjunction between whole sequences; parentheses make a sequence act like a single character for the operators around it. Operator precedence, from highest to lowest: parentheses; counters `* + ? {}`; sequences and anchors; disjunction `|`.


In [7]:
show(r"cat|dog", "raining cats and dogs")
show(r"gupp(y|ies)", "one guppy, two guppies")
show(r"guppy|ies", "one guppy, two guppies")
show(r"Column [0-9]+ *", "Column 1 Column 2 Column 3")
show(r"(Column [0-9]+ *)*", "Column 1 Column 2 Column 3")

r"cat|dog"   raining [cat]s and [dog]s   (2 matches)

r"gupp(y|ies)"   one [guppy], two [guppies]   (2 matches)

r"guppy|ies"   one [guppy], two gupp[ies]   (2 matches)

r"Column [0-9]+ *"   [Column 1 ][Column 2 ][Column 3]   (3 matches)

r"(Column [0-9]+ *)*"   [Column 1 Column 2 Column 3][]   (2 matches)

### Greedy and non-greedy

`*` and `+` are **greedy**: they match as much text as they can. Adding `?` after them (`*?`, `+?`) makes them non-greedy: as little as possible.


In [8]:
show(r"[a-z]*", "once upon a time")
show(r"<.*>", "<a> x <b>")
show(r"<.*?>", "<a> x <b>")

r"[a-z]*"   [once][] [upon][] [a][] [time][]   (8 matches)

r"<.*>"   [<a> x <b>]   (1 matches)

r"<.*?>"   [<a>] x [<b>]   (2 matches)

### A simple example: finding the article _the_

Each pattern fixes an error of the previous one. A **false positive** is a string we matched but should not have (it lowers precision); a **false negative** is a string we should have matched but missed (it lowers recall).


In [9]:
TEXT = "The other one there, the blithe one. Then the end: the"
show(r"the", TEXT)  # misses The: false negative
show(r"[tT]he", TEXT)  # other, there, blithe, Then: false positives
show(r"\b[tT]he\b", TEXT)  # a word boundary on both sides
show(
    r"[^a-zA-Z][tT]he[^a-zA-Z]", TEXT
)  # without \b: misses The at the start and the at the end
show(r"(^|[^a-zA-Z])[tT]he([^a-zA-Z]|$)", TEXT)  # the book's final version

r"the"   The o[the]r one [the]re, [the] bli[the] one. Then [the] end: [the]   (6 matches)

r"[tT]he"   [The] o[the]r one [the]re, [the] bli[the] one. [The]n [the] end: [the]   (8 matches)

r"\b[tT]he\b"   [The] other one there, [the] blithe one. Then [the] end: [the]   (4 matches)

r"[^a-zA-Z][tT]he[^a-zA-Z]"   The other one there,[ the ]blithe one. Then[ the ]end: the   (2 matches)

r"(^|[^a-zA-Z])[tT]he([^a-zA-Z]|$)"   [The ]other one there,[ the ]blithe one. Then[ the ]end:[ the]   (4 matches)

### Aliases and escapes

`\d` any digit, `\w` any alphanumeric character or underscore, `\s` whitespace; the upper-case versions negate them. Special characters are escaped with a backslash when meant literally: `\.`, `\*`, `\?`, `\\`.


In [10]:
show(r"\d+", "Party of 5, table 12")
show(r"\w+", "l'opéra, café_2 and naïve")
show(r"\s+", "tabs\tand   spaces\nand newlines")
show(r"\.", "K.A.P.L.A.N")
show(r"\?", "Would you light my candle?")

r"\d+"   Party of [5], table [12]   (2 matches)

r"\w+"   [l]'[opéra], [café_2] [and] [naïve]   (5 matches)

r"\s+"   tabs[	]and[   ]spaces[
]and[ ]newlines   (4 matches)

r"\."   K[.]A[.]P[.]L[.]A[.]N   (5 matches)

r"\?"   Would you light my candle[?]   (1 matches)

### Exercise 2 of the sheet (textbook exercise 2.4)

Here `check` has the test strings; the patterns below are **first attempts** that fail some of the tests. Find out why, fix each pattern, and run again until all tests pass. Then compare with your answer on paper.


In [11]:
# (a) all alphabetic strings
check(
    r"^[a-z]+$",
    accept=["hello", "Hello", "HLT", "b"],
    reject=["hello world", "abc1", "don't"],
)

r"^[a-z]+$"   (fullmatch)
  ok     should match      'hello'
  WRONG  should match      'Hello'
  WRONG  should match      'HLT'
  ok     should match      'b'
  ok     should not match  'hello world'
  ok     should not match  'abc1'
  ok     should not match  "don't"
  5 of 7 correct


In [12]:
# (b) all lower-case alphabetic strings ending in a b
check(
    r"^[a-z]+b$",
    accept=["b", "bob", "climb", "ab"],
    reject=["Bob", "bobs", "climB", "lamb chop"],
)

r"^[a-z]+b$"   (fullmatch)
  WRONG  should match      'b'
  ok     should match      'bob'
  ok     should match      'climb'
  ok     should match      'ab'
  ok     should not match  'Bob'
  ok     should not match  'bobs'
  ok     should not match  'climB'
  ok     should not match  'lamb chop'
  7 of 8 correct


In [13]:
# (c) all strings over {a, b} in which each a is immediately preceded by and immediately followed by a b
check(
    r"^(b|bab)*$",
    accept=["b", "bab", "bbabb", "babab", "bbb"],
    reject=["a", "ab", "ba", "baab", "abab"],
)

r"^(b|bab)*$"   (fullmatch)
  ok     should match      'b'
  ok     should match      'bab'
  ok     should match      'bbabb'
  WRONG  should match      'babab'
  ok     should match      'bbb'
  ok     should not match  'a'
  ok     should not match  'ab'
  ok     should not match  'ba'
  ok     should not match  'baab'
  ok     should not match  'abab'
  9 of 10 correct


### Exercise 3 of the sheet (textbook exercise 2.5)

Same game: a first attempt for each item, to be fixed. Here a match anywhere in the string is enough (`how="search"`). A _word_ is an alphabetic string separated from other words by whitespace, punctuation or line breaks.


In [14]:
# (a) two consecutive repeated words
check(
    r"(\w+) \1",
    how="search",
    accept=["Humbert Humbert", "the the", "I saw the the cat"],
    reject=["the bug", "the big bug", "in the theatre", "this is"],
)

r"(\w+) \1"   (search)
  ok     should match      'Humbert Humbert'
  ok     should match      'the the'
  ok     should match      'I saw the the cat'
  ok     should not match  'the bug'
  ok     should not match  'the big bug'
  WRONG  should not match  'in the theatre'
  WRONG  should not match  'this is'
  5 of 7 correct


In [15]:
# (b) lines that start with an integer and end with a word
check(
    r"^\d+.*[A-Za-z]+$",
    how="search",
    accept=["42 is the answer", "7 samurai", "2026 was a good year"],
    reject=["The 42 answers", "42nd street", "42 is the answer.", "42 44"],
)

r"^\d+.*[A-Za-z]+$"   (search)
  ok     should match      '42 is the answer'
  ok     should match      '7 samurai'
  ok     should match      '2026 was a good year'
  ok     should not match  'The 42 answers'
  WRONG  should not match  '42nd street'
  ok     should not match  '42 is the answer.'
  ok     should not match  '42 44'
  6 of 7 correct


In [16]:
# (c) strings that contain both the word grotto and the word raven
check(
    r"grotto.*raven",
    how="search",
    accept=["the raven flew into the grotto", "a grotto with a raven"],
    reject=["grottos and ravens", "a grotto", "the raven", "grottoraven"],
)

r"grotto.*raven"   (search)
  WRONG  should match      'the raven flew into the grotto'
  ok     should match      'a grotto with a raven'
  WRONG  should not match  'grottos and ravens'
  ok     should not match  'a grotto'
  ok     should not match  'the raven'
  WRONG  should not match  'grottoraven'
  3 of 6 correct


### Exercises 5 and 6 of the sheet

Exercise 5: false positives and false negatives. Exercise 6: precedence and greediness; the first highlighted match is the answer to the questions about the first match.


In [17]:
SENT = "The theory of the other them: the end."
show(r"[tT]he", SENT)
show(r"\b[tT]he\b", SENT)
show(r"\bthe\b", SENT)

r"[tT]he"   [The] [the]ory of [the] o[the]r [the]m: [the] end.   (6 matches)

r"\b[tT]he\b"   [The] theory of [the] other them: [the] end.   (3 matches)

r"\bthe\b"   The theory of [the] other them: [the] end.   (2 matches)

In [18]:
show(r"the*", "thethe theee")
show(r"gupp(y|ies)", "guppies")
show(r"guppy|ies", "guppies")
show(r"<.*>", "<a> x <b>")
show(r"<.*?>", "<a> x <b>")

r"the*"   [the][the] [theee]   (3 matches)

r"gupp(y|ies)"   [guppies]   (1 matches)

r"guppy|ies"   gupp[ies]   (1 matches)

r"<.*>"   [<a> x <b>]   (1 matches)

r"<.*?>"   [<a>] x [<b>]   (2 matches)

## 3. Substitutions, capture groups and lookahead

`re.sub(pattern, repl, string)` replaces every match of the pattern in the string. Parentheses **capture** what they match in numbered groups, counted from the left; in the replacement, and later in the same pattern, `\1`, `\2`, ... refer back to them.


In [19]:
print(re.sub(r"cherry", r"apricot", "I like cherry pie and cherry tart"))
print(re.sub(r"janet", r"Janet", "janet and janet's cat"))
print(re.sub(r"(\d{2})/(\d{2})/(\d{4})", r"\2-\1-\3", "The date is 10/15/2011"))
show(r"\b([A-Za-z]+)\s+\1\b", "Paris in the the spring, and and so on")

I like apricot pie and apricot tart
Janet and Janet's cat
The date is 15-10-2011


r"\b([A-Za-z]+)\s+\1\b"   Paris in [the the] spring, [and and] so on   (2 matches)

When we need grouping but not capturing, `(?: ... )` is a **non-capturing group**. Below, the first 14 dates of a line are skipped without being stored, and only the 15th is captured in group 1.


In [20]:
dates = " ".join(f"{d:02d}/01/2026" for d in range(1, 21))
m = re.search(r"(?:\d\d/\d\d/\d{4}\s+){14}(\d\d/\d\d/\d{4})", dates)
print(m.group(1))

15/01/2026


**Lookahead** assertions look at what comes next without consuming it: `(?= ... )` is true if the pattern follows, `(?! ... )` if it does not. Equation 2.18 of the book captures the first word of a line, but only if it does not start with _t_ or _T_.


In [21]:
for line in ["The cat sleeps", "Cats sleep", "tomorrow, maybe", "Maybe tomorrow"]:
    m = re.search(r"^(?![tT])(\w+)\b", line)
    print(f"{line!r:20}", m.group(1) if m else None)
show(r"\d+(?=kg)", "5kg of apples, 12 m of rope, 7kg of flour")

'The cat sleeps'     None
'Cats sleep'         Cats
'tomorrow, maybe'    None
'Maybe tomorrow'     Maybe


r"\d+(?=kg)"   [5]kg of apples, 12 m of rope, [7]kg of flour   (2 matches)

### Exercise 4 of the sheet

(a) runs the substitution of the sheet. For (b) the cell below has a first attempt that puts brackets around every _digit_; change the pattern so that it brackets every integer.


In [22]:
print(re.sub(r"(\d{2})/(\d{2})/(\d{4})", r"\2-\1-\3", "Due 03/09/2026 and 12/25/2025"))
print(
    re.sub(r"(\d)", r"<\1>", "the 35 boxes weigh 120 kg")
)  # first attempt for (b): fix it

Due 09-03-2026 and 25-12-2025
the <3><5> boxes weigh <1><2><0> kg


## 4. ELIZA, a cascade of substitutions

ELIZA (Weizenbaum 1966) works with a cascade of regex substitutions. After the input is uppercased, a first set of substitutions changes the point of view: MY becomes YOUR, I'M becomes YOU ARE, and so on. A second set matches patterns in the input and turns it into a complete response; the two patterns below are the book's (page 55). If no pattern applies, this toy version echoes the transformed input.

The order matters: the first set runs before the second, so the second can look for YOU ARE.


In [23]:
POINT_OF_VIEW = [  # first set: change the point of view
    (r"\bMY\b", "YOUR"),
    (r"\bI['’]M\b", "YOU ARE"),
    (r"\bME\b", "YOU"),
    (r"\bMYSELF\b", "YOURSELF"),
]
RESPONSES = [  # second set: a pattern that matches turns the input into a response
    (r".* YOU ARE (DEPRESSED|SAD) .*", r"I AM SORRY TO HEAR YOU ARE \1"),
    (r".* ALWAYS .*", r"CAN YOU THINK OF A SPECIFIC EXAMPLE"),
]


def eliza(utterance, verbose=False):
    s = utterance.upper()
    for pattern, repl in POINT_OF_VIEW:
        s = re.sub(pattern, repl, s)
    s = re.sub(r"^(WELL|OH|SO),?\s+", "", s)  # drop an initial interjection
    s = re.sub(r"[.!?]+$", "", s).strip()  # and the final punctuation
    if verbose:
        print("   after the first set:", s)
    for pattern, repl in RESPONSES:
        if re.match(
            pattern, s + " "
        ):  # a space at the end, so that ' .*' can match at the end
            return re.sub(pattern, repl, s + " ").strip()
    return s


for user in [
    "They're always bugging us about something or other.",
    "Well, my boyfriend made me come here.",
    "He says I'm depressed much of the time.",
]:
    print("User: ", user)
    print("ELIZA:", eliza(user, verbose=True))

User:  They're always bugging us about something or other.
   after the first set: THEY'RE ALWAYS BUGGING US ABOUT SOMETHING OR OTHER
ELIZA: CAN YOU THINK OF A SPECIFIC EXAMPLE
User:  Well, my boyfriend made me come here.
   after the first set: YOUR BOYFRIEND MADE YOU COME HERE
ELIZA: YOUR BOYFRIEND MADE YOU COME HERE
User:  He says I'm depressed much of the time.
   after the first set: HE SAYS YOU ARE DEPRESSED MUCH OF THE TIME
ELIZA: I AM SORRY TO HEAR YOU ARE DEPRESSED


Textbook exercise 2.6 asks you to implement an ELIZA-like program, possibly in another domain where simple repetition makes sense. Start from here: add rules to `RESPONSES` (and to `POINT_OF_VIEW`), for example the two below, and try your own sentences. Where does the cascade break?


In [24]:
RESPONSES.append((r"^YOU ARE (.*)", r"HOW LONG HAVE YOU BEEN \1"))
RESPONSES.append((r".* YOUR (MOTHER|FATHER) .*", r"TELL ME MORE ABOUT YOUR FAMILY"))
for user in [
    "I'm tired of this course.",
    "I think my mother likes regular expressions.",
    "The weather is nice.",
]:
    print(f"{user:48} -> {eliza(user)}")

I'm tired of this course.                        -> HOW LONG HAVE YOU BEEN TIRED OF THIS COURSE
I think my mother likes regular expressions.     -> TELL ME MORE ABOUT YOUR FAMILY
The weather is nice.                             -> THE WEATHER IS NICE


## 5. The GPT-2 pre-tokenizer

Before BPE runs, the corpus is pre-tokenized with a regular expression, so that tokens never cross word boundaries (Figure 2.15 of the book). The original pattern uses `\p{L}` (any Unicode letter) and `\p{N}` (any number), which need the external `regex` module. With the standard `re` module we can write the same classes: `[^\W\d_]` is a letter (a word character that is neither a digit nor the underscore), `\d` a digit, `(?:[^\s\w]|_)` a character that is neither a space nor a letter or digit.


In [25]:
GPT2_RE = (
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+"""
)


def pretokenize(text):
    return re.findall(GPT2_RE, text)


print(pretokenize("We're 350 dogs! Um, lunch?"))
show(GPT2_RE, "We're 350 dogs! Um, lunch?")

['We', "'re", ' 350', ' dogs', '!', ' Um', ',', ' lunch', '?']


r"'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+"   [We]['re][ 350][ dogs][!][ Um][,][ lunch][?]   (9 matches)

Some tokens start with a space (` dogs`), others do not (`We`, `!`): the space goes with the word that follows it. Contractions become tokens of their own.

**Exercise 7 of the sheet.** Predict the chunks on paper, then run:


In [26]:
print(pretokenize("I'm 42 years old!! Don't ask."))
print(
    pretokenize("Don’t ask.")
)  # with a typographic apostrophe the contraction rule does not fire

['I', "'m", ' 42', ' years', ' old', '!!', ' Don', "'t", ' ask', '.']
['Don', '’', 't', ' ask', '.']


**Optional.** With the `regex` module (`pip install regex`) you can run the original pattern of the book and check that it gives the same chunks.


In [27]:
import regex

GPT2_ORIGINAL = (
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)
for text in [
    "We're 350 dogs! Um, lunch?",
    "I'm 42 years old!! Don't ask.",
    "Città di Pisa: 1.000.000 abitanti!!!",
]:
    original = regex.findall(GPT2_ORIGINAL, text)
    print(original, "same as re:", original == pretokenize(text))

['We', "'re", ' 350', ' dogs', '!', ' Um', ',', ' lunch', '?'] same as re: True
['I', "'m", ' 42', ' years', ' old', '!!', ' Don', "'t", ' ask', '.'] same as re: True
['Città', ' di', ' Pisa', ':', ' 1', '.', '000', '.', '000', ' abitanti', '!!!'] same as re: True


## 6. Word counts, the Unix way

Section 2.7 of the book counts the words of Shakespeare with one Unix command line:

`tr -sc 'A-Za-z' '\n' < sh.txt | tr A-Z a-z | sort | uniq -c | sort -n -r`

`tr -sc` turns every run of non-letters into a single newline (one word per line), `tr A-Z a-z` lower-cases, `sort | uniq -c` collapses and counts identical lines, `sort -n -r` orders by count, highest first. The same steps in Python, on a corpus that needs no download: the documentation strings of some modules of the Python standard library (the exact counts depend on your Python version).


In [28]:
import importlib, inspect
from collections import Counter

MODULES = """os re json collections itertools random statistics textwrap datetime pathlib logging argparse csv math
functools string heapq bisect difflib shutil glob tempfile time calendar decimal fractions operator copy pprint enum
typing dataclasses contextlib unittest zipfile gzip hashlib struct codecs io threading queue subprocess socket email
html sqlite3 pickle configparser platform inspect ast tokenize traceback locale gettext shlex uuid base64 array types""".split()


def module_text(name):
    """the docstring of a module and of its public functions and classes"""
    mod = importlib.import_module(name)
    parts = [inspect.getdoc(mod) or ""]
    for member, obj in inspect.getmembers(mod):
        if not member.startswith("_") and (
            inspect.isfunction(obj) or inspect.isclass(obj)
        ):
            parts.append(inspect.getdoc(obj) or "")
    return "\n".join(parts)


corpus = "\n".join(module_text(m) for m in MODULES)


def unix_word_counts(text):
    words = re.sub(r"[^A-Za-z]+", "\n", text).split("\n")  # tr -sc 'A-Za-z' '\n'
    words = [w.lower() for w in words if w]  # tr A-Z a-z
    return Counter(words).most_common()  # sort | uniq -c | sort -n -r


counts = unix_word_counts(corpus)
print(f"{sum(n for _, n in counts)} words, {len(counts)} distinct")
for word, n in counts[:12]:
    print(f"{n:6d} {word}")

49321 words, 3772 distinct
  2893 the
  1613 a
  1137 is
  1033 to
  1003 of
   776 and
   676 for
   542 be
   534 if
   532 in
   397 or
   397 are


As in Shakespeare, the most frequent words are the short function words: articles, prepositions, pronouns. Handy for quick statistics on any English text; for anything more complex we use the tokenization algorithms seen so far.

**Optional.** On macOS, Linux or Colab, the real command line (the `!` hands the line to the shell). The counts are close to the Python ones: `tr` treats accented letters and the underscore like any other non-letter, as the Python version does.


In [29]:
with open("docstrings.txt", "w", encoding="utf-8") as f:
    f.write(corpus)
!tr -sc 'A-Za-z' '\n' < docstrings.txt | tr A-Z a-z | sort | uniq -c | sort -n -r | head -12

   2893 the
   1613 a
   1137 is
   1033 to
   1003 of
    776 and
    676 for
    542 be
    534 if
    532 in
    397 or
    397 are
sort: write non riuscita: 'standard output': Pipe interrotta
sort: errore di scrittura


**Optional.** The complete works of Shakespeare from Project Gutenberg (about 5 MB). The counts are of the same order as the book's (27,378 _the_, 26,084 _and_, 22,538 _i_), not identical, since the edition differs.


In [ ]:
import urllib.request

raw = (
    urllib.request
    .urlopen("https://www.gutenberg.org/cache/epub/100/pg100.txt", timeout=60)
    .read()
    .decode("utf-8")
)
shakespeare = raw.split("*** START OF")[1].split("*** END OF")[0]
for word, n in unix_word_counts(shakespeare)[:10]:
    print(f"{n:6d} {word}")

URLError: <urlopen error [Errno 111] Connection refused>

## 7. Rule-based tokenization

### A regular expression tokenizer

Figure 2.16 of the book: the pattern that NLTK's `regexp_tokenize` uses in Chapter 3 of Bird et al. (2009). With the verbose flag `(?x)` Python ignores the whitespace and the comments in the pattern; `re.findall` returns the tokens, as `nltk.regexp_tokenize` does.


In [ ]:
NLTK_PATTERN = r"""(?x)        # set flag to allow verbose regexps
      (?:[A-Z]\.)+             # abbreviations, e.g. U.S.A.
    | \w+(?:-\w+)*             # words with optional internal hyphens
    | \$?\d+(?:\.\d+)?%?       # currency, percentages, e.g. $12.40, 82%
    | \.\.\.                   # ellipsis
    | [][.,;"'?():_`-]         # these are separate tokens; includes ], [
"""
print(re.findall(NLTK_PATTERN, "That U.S.A. poster-print costs $12.40..."))
print(re.findall(NLTK_PATTERN, "It costs 12.40 euros, 82% of $15."))

['That', 'U.S.A.', 'poster-print', 'costs', '$12.40', '...']
['It', 'costs', '12', '.', '40', 'euros', ',', '82', 'of', '$15', '.']


The second line shows that the order of the alternatives matters: at each position the first alternative that matches wins. Without the dollar sign, `\w+` takes `12` before the number alternative is tried, so `12.40` is split in three and the `%` of `82%` is lost. Moving the number alternative before the word alternative fixes it: try.

### The Penn Treebank standard

The Penn Treebank tokenizer was originally a script of substitutions run one after the other: a cascade, like ELIZA. The simplified version below separates clitics (_doesn't_ becomes _does n't_), keeps hyphenated words and numbers together, and separates out all punctuation. It assumes one sentence per line, so only the final period is split off.


In [ ]:
def ptb_tokenize(sentence):
    s = sentence
    s = re.sub(r'"', ' " ', s)  # quotes
    s = re.sub(r"([;:?!()\[\]])", r" \1 ", s)  # punctuation that is always a token
    s = re.sub(
        r",(?!\d)|(?<!\d),", " , ", s
    )  # commas, but not inside numbers like 555,500
    s = re.sub(r"\$", " $ ", s)  # the dollar sign
    s = re.sub(r"\.(?=[\s\"')\]]*$)", " . ", s)  # the final period only
    s = re.sub(r"(?i)n't\b", " n't", s)  # clitics: n't first
    s = re.sub(r"(?i)'(s|re|ve|ll|d|m)\b", r" '\1", s)  # then 's 're 've 'll 'd 'm
    return s.split()


book = ptb_tokenize(
    '"The San Francisco-based restaurant," they said, "doesn\'t charge $10".'
)
print(" ".join(book))

" The San Francisco-based restaurant , " they said , " does n't charge $ 10 " .


**Exercise 8 of the sheet.** Tokenize on paper and count, then check.


In [ ]:
tokens = ptb_tokenize(
    '"They don\'t know," she said, "it\'s $12.50 per hour-long lesson."'
)
print(" ".join(tokens))
print(len(tokens), "tokens")

" They do n't know , " she said , " it 's $ 12.50 per hour-long lesson . "
20 tokens


### Sentence segmentation

Question marks and exclamation points are fairly unambiguous sentence boundaries; the period is not: it also marks abbreviations (_Dr._, _Inc._) and decimal numbers. Splitting after every period followed by a space breaks _Dr. Rossi_ in two.


In [ ]:
TEXT9 = (
    "Dr. Rossi arrived at 9 a.m. He works for Acme Inc. The meeting lasted 2.5 hours."
)
for s in re.split(r"(?<=[.!?])\s+", TEXT9):
    print("|", s)

| Dr.
| Rossi arrived at 9 a.m.
| He works for Acme Inc.
| The meeting lasted 2.5 hours.


A slightly better rule, deciding for each period whether it belongs to a token: a token ending in a period ends a sentence if it is not a title (_Dr._, _Mr._, ...) and the next word is capitalized, or if it is the last token. The final period of an abbreviation like _a.m._ or _Inc._ can then be both part of the token and the end of the sentence, as in exercise 9 of the sheet. In practice the list of abbreviations is built by hand or learned from data (Kiss and Strunk 2006).


In [ ]:
TITLES = {"Dr.", "Mr.", "Mrs.", "Ms.", "Prof.", "St."}


def segment(text):
    tokens = text.split()
    sentences, current = [], []
    for k, tok in enumerate(tokens):
        current.append(tok)
        nxt = tokens[k + 1] if k + 1 < len(tokens) else None
        if tok[-1] in "!?" or (
            tok.endswith(".")
            and tok not in TITLES
            and (nxt is None or nxt[0].isupper())
        ):
            sentences.append(" ".join(current))
            current = []
    if current:
        sentences.append(" ".join(current))
    return sentences


for s in segment(TEXT9):
    print("|", s)
print()
for s in segment(
    "She moved to the U.S. Army base. It was 9 p.m. when she arrived."
):  # where the rule fails
    print("|", s)

| Dr. Rossi arrived at 9 a.m.
| He works for Acme Inc.
| The meeting lasted 2.5 hours.

| She moved to the U.S.
| Army base.
| It was 9 p.m. when she arrived.


## 8. Minimum edit distance

The minimum edit distance between two strings is the minimum cost of the insertions, deletions and substitutions that turn one into the other. The table D has one row per character of the source X (length n) and one column per character of the target Y (length m), plus a row and a column for the empty string #. D[i, j] is the distance between the first i characters of X and the first j characters of Y, and D[n, m] is the answer.

Textbook exercise 2.9 asks you to implement the algorithm of Figure 2.21 and to check it against your hand computations: try it yourself before reading the next cell.


In [ ]:
def min_edit_distance(source, target, ins_cost=1, del_cost=1, sub_cost=2):
    """The algorithm of Figure 2.21. Returns the whole table D."""
    n, m = len(source), len(target)
    D = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):  # zeroth column: deletions only
        D[i][0] = D[i - 1][0] + del_cost
    for j in range(1, m + 1):  # zeroth row: insertions only
        D[0][j] = D[0][j - 1] + ins_cost
    for i in range(1, n + 1):  # recurrence, row by row
        for j in range(1, m + 1):
            sub = 0 if source[i - 1] == target[j - 1] else sub_cost
            D[i][j] = min(
                D[i - 1][j] + del_cost,  # from above: delete source[i]
                D[i - 1][j - 1]
                + sub,  # from the diagonal: substitute, or match at cost 0
                D[i][j - 1] + ins_cost,
            )  # from the left: insert target[j]
    return D


def print_table(source, target, D, path=()):
    """Rows: #, then the source; columns: #, then the target. Cells on the path in brackets."""
    rows, cols = ["#"] + list(source), ["#"] + list(target)
    w = max(5, max(len(c) for c in cols) + 2)  # column width (wider for words)
    rw = max(len(r) for r in rows) + 2
    print(" " * rw + "".join(f"{c:>{w}}" for c in cols))
    for i, r in enumerate(rows):
        cells = "".join(
            f"{f'[{v}]' if (i, j) in path else str(v):>{w}}" for j, v in enumerate(D[i])
        )
        print(f"{r:>{rw}}" + cells)


D = min_edit_distance("intention", "execution")
print_table("intention", "execution", D)
print("\ndistance, substitutions at 2:", D[-1][-1])
print(
    "distance, substitutions at 1:",
    min_edit_distance("intention", "execution", sub_cost=1)[-1][-1],
)

       #    e    x    e    c    u    t    i    o    n
  #    0    1    2    3    4    5    6    7    8    9
  i    1    2    3    4    5    6    7    6    7    8
  n    2    3    4    5    6    7    8    7    8    7
  t    3    4    5    6    7    8    7    8    9    8
  e    4    3    4    5    6    7    8    9   10    9
  n    5    4    5    6    7    8    9   10   11   10
  t    6    5    6    7    8    9    8    9   10   11
  i    7    6    7    8    9   10    9    8    9   10
  o    8    7    8    9   10   11   10    9    8    9
  n    9    8    9   10   11   12   11   10    9    8

distance, substitutions at 2: 8
distance, substitutions at 1: 5


The table is Figure 2.20 of the book: distance 8 with substitutions at cost 2 (Levenshtein's alternative version), 5 with all costs at 1.

### The backtrace

Textbook exercise 2.10: to recover an alignment, store in each cell a pointer to the cell (or cells) it came from, then follow the pointers back from the last cell to the first. Every complete path is a minimum-cost alignment. When several pointers tie, the code below prefers the diagonal, then the cell above, then the cell to the left; other choices give other alignments with the same cost.


In [ ]:
def backtrace(source, target, ins_cost=1, del_cost=1, sub_cost=2):
    D = min_edit_distance(source, target, ins_cost, del_cost, sub_cost)
    i, j = len(source), len(target)
    path, top, bottom, ops = [(i, j)], [], [], []
    while i > 0 or j > 0:
        sub = sub_cost if i > 0 and j > 0 and source[i - 1] != target[j - 1] else 0
        if i > 0 and j > 0 and D[i][j] == D[i - 1][j - 1] + sub:  # diagonal
            top.append(source[i - 1])
            bottom.append(target[j - 1])
            ops.append("s" if sub else " ")
            i, j = i - 1, j - 1
        elif i > 0 and D[i][j] == D[i - 1][j] + del_cost:  # from above: deletion
            top.append(source[i - 1])
            bottom.append("*")
            ops.append("d")
            i -= 1
        else:  # from the left: insertion
            top.append("*")
            bottom.append(target[j - 1])
            ops.append("i")
            j -= 1
        path.append((i, j))
    return D, path, top[::-1], bottom[::-1], ops[::-1]


def show_alignment(source, target, table=True, **costs):
    D, path, top, bottom, ops = backtrace(source, target, **costs)
    if table:
        print_table(source, target, D, set(path))
    print(f"\ndistance {D[-1][-1]}")
    widths = [max(len(a), len(b)) for a, b in zip(top, bottom)]
    for row in (top, bottom, ops):
        print("   " + " ".join(f"{x:<{w}}" for x, w in zip(row, widths)))
    print()


show_alignment("intention", "execution")

       #    e    x    e    c    u    t    i    o    n
  #  [0]    1    2    3    4    5    6    7    8    9
  i  [1]    2    3    4    5    6    7    6    7    8
  n    2  [3]    4    5    6    7    8    7    8    7
  t    3    4  [5]    6    7    8    7    8    9    8
  e    4    3    4  [5]  [6]    7    8    9   10    9
  n    5    4    5    6    7  [8]    9   10   11   10
  t    6    5    6    7    8    9  [8]    9   10   11
  i    7    6    7    8    9   10    9  [8]    9   10
  o    8    7    8    9   10   11   10    9  [8]    9
  n    9    8    9   10   11   12   11   10    9  [8]

distance 8
   i n t e * n t i o n
   * e x e c u t i o n
   d s s   i s        



The alignment is the one of Figure 2.17: `d` deletion, `i` insertion, `s` substitution, a blank for a match.

Your backtrace on paper may follow other arrows and still be right. The function below follows **all** the pointers and lists every minimum-cost alignment, so you can find yours among them.


In [ ]:
def all_alignments(source, target, ins_cost=1, del_cost=1, sub_cost=2, limit=30):
    D = min_edit_distance(source, target, ins_cost, del_cost, sub_cost)
    found = []

    def walk(i, j, top, bottom, ops):  # follow every pointer back to the origin
        if len(found) >= limit:
            return
        if i == 0 and j == 0:
            found.append((top[::-1], bottom[::-1], ops[::-1]))
            return
        if i > 0 and j > 0:
            sub = 0 if source[i - 1] == target[j - 1] else sub_cost
            if D[i][j] == D[i - 1][j - 1] + sub:
                walk(
                    i - 1,
                    j - 1,
                    top + [source[i - 1]],
                    bottom + [target[j - 1]],
                    ops + ["s" if sub else " "],
                )
        if i > 0 and D[i][j] == D[i - 1][j] + del_cost:
            walk(i - 1, j, top + [source[i - 1]], bottom + ["*"], ops + ["d"])
        if j > 0 and D[i][j] == D[i][j - 1] + ins_cost:
            walk(i, j - 1, top + ["*"], bottom + [target[j - 1]], ops + ["i"])

    walk(len(source), len(target), [], [], [])
    return found


def print_all_alignments(source, target, **costs):
    found = all_alignments(source, target, **costs)
    print(
        f"{len(found)} minimum-cost alignment{'s' if len(found) > 1 else ''} of {source} and {target}:"
    )
    for top, bottom, ops in found:
        print()
        for row in (top, bottom, ops):
            print("   " + " ".join(row))


print(
    len(all_alignments("intention", "execution", limit=10**6)),
    "minimum-cost alignments of intention and execution",
)
print_all_alignments("cat", "cut")

134 minimum-cost alignments of intention and execution
3 minimum-cost alignments of cat and cut:

   c a t
   c u t
     s  

   c * a t
   c u * t
     i d  

   c a * t
   c * u t
     d i  


### Exercises 10 to 12 of the sheet

Fill the tables on paper, then check. Exercise 10 (textbook 2.7) and the first part of exercise 11 use cost 1 for substitutions too.


In [ ]:
show_alignment("leda", "deal", sub_cost=1)  # exercise 10
print_all_alignments("leda", "deal", sub_cost=1)

       #    d    e    a    l
  #  [0]    1    2    3    4
  l    1  [1]    2    3    3
  e    2    2  [1]    2    3
  d    3    2    2  [2]    3
  a    4    3    3    2  [3]

distance 3
   l e d a
   d e a l
   s   s s

2 minimum-cost alignments of leda and deal:

   l e d a
   d e a l
   s   s s

   l e d a *
   d e * a l
   s   d   i


In [ ]:
for sub in (1, 2):  # exercise 11 (textbook 2.8)
    d_brief = min_edit_distance("drive", "brief", sub_cost=sub)[-1][-1]
    d_divers = min_edit_distance("drive", "divers", sub_cost=sub)[-1][-1]
    print(f"substitution cost {sub}: drive-brief {d_brief}, drive-divers {d_divers}")
print()
show_alignment("drive", "brief", sub_cost=2)
show_alignment("drive", "divers", sub_cost=2)

substitution cost 1: drive-brief 3, drive-divers 3
substitution cost 2: drive-brief 4, drive-divers 3

       #    b    r    i    e    f
  #  [0]    1    2    3    4    5
  d    1  [2]    3    4    5    6
  r    2    3  [2]    3    4    5
  i    3    4    3  [2]    3    4
  v    4    5    4  [3]    4    5
  e    5    6    5    4  [3]  [4]

distance 4
   d r i v e *
   b r i * e f
   s     d   i

       #    d    i    v    e    r    s
  #  [0]    1    2    3    4    5    6
  d    1  [0]    1    2    3    4    5
  r    2  [1]    2    3    4    3    4
  i    3    2  [1]    2    3    4    5
  v    4    3    2  [1]    2    3    4
  e    5    4    3    2  [1]  [2]  [3]

distance 3
   d r i v e * *
   d * i v e r s
     d       i i



In [ ]:
show_alignment("sunday", "saturday")  # exercise 12, substitutions at 2
print_all_alignments("sunday", "saturday")

       #    s    a    t    u    r    d    a    y
  #  [0]    1    2    3    4    5    6    7    8
  s    1  [0]  [1]  [2]    3    4    5    6    7
  u    2    1    2    3  [2]    3    4    5    6
  n    3    2    3    4    3  [4]    5    6    7
  d    4    3    4    5    4    5  [4]    5    6
  a    5    4    3    4    5    6    5  [4]    5
  y    6    5    4    5    6    7    6    5  [4]

distance 4
   s * * u n d a y
   s a t u r d a y
     i i   s      

3 minimum-cost alignments of sunday and saturday:

   s * * u n d a y
   s a t u r d a y
     i i   s      

   s * * u * n d a y
   s a t u r * d a y
     i i   i d      

   s * * u n * d a y
   s a t u * r d a y
     i i   d i      


### Where edit distance is used

Spelling correction: which candidate is closest to what the user typed? The same code works on sequences of words instead of characters: the word error rate of a speech recognizer is the edit distance between the recognized words and the reference transcript, divided by the length of the reference; and two mentions of the same entity differ by a few words.


In [ ]:
typed = "graffe"
for cand in ["giraffe", "grail", "graf", "gaffe", "graph"]:
    print(f"{typed} -> {cand:8} {min_edit_distance(typed, cand, sub_cost=1)[-1][-1]}")
print()
reference = "the cat sat on the mat".split()
recognized = "the cat sat on mat today".split()
d = min_edit_distance(reference, recognized, sub_cost=1)[-1][-1]
print(f"word error rate = {d} / {len(reference)} = {d / len(reference):.2f}")
print()
show_alignment(
    "Stanford President Marc Tessier-Lavigne".split(),
    "Stanford University President Marc Tessier-Lavigne".split(),
    table=False,
)

graffe -> giraffe  1
graffe -> grail    3
graffe -> graf     2
graffe -> gaffe    1
graffe -> graph    3

word error rate = 2 / 6 = 0.33


distance 1
   Stanford *          President Marc Tessier-Lavigne
   Stanford University President Marc Tessier-Lavigne
            i                                        



## 9. What to do next

- The exercise sheet of this lecture: every exercise has a cell above to check your answer.
- Textbook exercises 2.4 to 2.10 (Chapter 2); 2.6, 2.9 and 2.10 are programming exercises, and sections 4 and 8 of this notebook are a starting point for them.
- Change the inputs: your own patterns and strings in `show` and `check`, new ELIZA rules, your own text for the word counts, pairs of words for the edit distance.
